In [ ]:
import os
from supabase import create_client
from dotenv import load_dotenv
import pandas as pd

## Supabase Load in Test table

In [22]:
# ─── 0) Load env & init client ───────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")  # need service key for writes
if not SUPABASE_URL or not SUPABASE_KEY:
    print("❌ SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY must be set in your .env")
    sys.exit(1)

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

### TEST
data = supabase.table("test_table").select("*").execute()
print(data.data)

[{'id': 1, 'name': 'Alpha', 'value': 10}, {'id': 2, 'name': 'Bravo', 'value': 20}, {'id': 3, 'name': 'Charlie', 'value': 30}, {'id': 4, 'name': 'Delta', 'value': 40}, {'id': 5, 'name': 'Echo', 'value': 50}, {'id': 6, 'name': 'Foxtrot', 'value': 60}, {'id': 7, 'name': 'Golf', 'value': 70}, {'id': 8, 'name': 'Hotel', 'value': 80}, {'id': 9, 'name': 'India', 'value': 90}, {'id': 10, 'name': 'Juliet', 'value': 100}]


## Creating Tables in Supabase 

Since supabase-py implements the PostgREST HTTP API, only certain types of SQL operations can be run remotely, mainly SELECT, INSERT, UPDATE, DELETE statements. 

Therefore, the following cells will generate SQL queries that must be run in the Supabase Dashboard SQL editor itself. These are the queries that are used to create the tables. 

In [18]:
quiz_df = pd.read_csv('quiz_df.csv')
print(quiz_df.head())
print(quiz_df.info())

  question_id quiz_id                                      question_text  \
0         QN1    QZ01  How does exploratory data analysis enhance dec...   
1         QN1    QZ01  How does exploratory data analysis enhance dec...   
2         QN1    QZ01  How does exploratory data analysis enhance dec...   
3        QN10    QZ01  How can anomaly detection in datasets improve ...   
4        QN10    QZ01  How can anomaly detection in datasets improve ...   

                  subtopic student_id module_id  \
0  Intro to Data Analytics        S01      DAVA   
1  Intro to Data Analytics        S01      DAVA   
2  Intro to Data Analytics        S01      DAVA   
3  Intro to Data Analytics        S01      DAVA   
4  Intro to Data Analytics        S01      DAVA   

                      module_name  attempt_number  quiz_score_percent  \
0  Data Visualization & Analytics               1                  40   
1  Data Visualization & Analytics               2                  30   
2  Data Visualiza

Based on the following table schema (quiz_df.info()), we can write the following SQL query to create the table in Supabase:
```
CREATE TABLE IF NOT EXISTS quiz_df (
  question_id          TEXT        NOT NULL,
  quiz_id              TEXT        NOT NULL,
  question_text        TEXT        NOT NULL,
  subtopic             TEXT        NOT NULL,
  student_id           TEXT        NOT NULL,
  module_id            TEXT        NOT NULL,
  module_name          TEXT        NOT NULL,
  attempt_number       INTEGER     NOT NULL,
  quiz_score_percent   INTEGER     NOT NULL,
  quiz_timestamp       TIMESTAMPTZ NOT NULL,
  quiz_duration_secs   INTEGER     NOT NULL,
  is_correct           BOOLEAN     NOT NULL
);

SELECT * FROM quiz_df;
```

In [21]:
test_df = pd.read_csv('test_df.csv')
print(test_df.head())
print(test_df.info())

  question_id test_id                 subtopic  \
0         QN1    TS01  Intro to Data Analytics   
1         QN1    TS01  Intro to Data Analytics   
2         QN1    TS01  Intro to Data Analytics   
3         QN1    TS01  Intro to Data Analytics   
4         QN1    TS01  Intro to Data Analytics   

                                       question_text student_id module_id  \
0  How can anomaly detection in data analytics tr...        S01      DAVA   
1  How can anomaly detection in data analytics tr...        S02      DAVA   
2  How can anomaly detection in data analytics tr...        S03      DAVA   
3  How can anomaly detection in data analytics tr...        S04      DAVA   
4  How can anomaly detection in data analytics tr...        S05      DAVA   

                      module_name  test_score_percent       test_timestamp  \
0  Data Visualization & Analytics                  76  2024-10-17 20:31:13   
1  Data Visualization & Analytics                  64  2024-01-11 19:10:02   
2 

```
CREATE TABLE IF NOT EXISTS test_df (
  question_id         TEXT        NOT NULL,
  test_id             TEXT        NOT NULL,
  subtopic            TEXT        NOT NULL,
  question_text       TEXT        NOT NULL,
  student_id          TEXT        NOT NULL,
  module_id           TEXT        NOT NULL,
  module_name         TEXT        NOT NULL,
  test_score_percent  INTEGER     NOT NULL,
  test_timestamp      TIMESTAMPTZ NOT NULL,
  is_correct          BOOLEAN     NOT NULL
);

SELECT * FROM test_df;
```


## Loading data into Supabase tables

We will now batch-load in data from quiz_df csv file into Supabase table quiz_df. The function reload_table_from_csv takes in csv path and loads it into the specified table name in Supabase. 

In [43]:
def reload_table_from_csv(TABLE: str, CSV_PATH: str) -> int:
    resp = supabase.table(TABLE).select("*", count="exact").limit(1).execute()

    if resp.data is None:
        raise RuntimeError(f"Table `{TABLE}` does not exist; define its schema first.")

    # clear the table if it's not empty
    if (resp.count or 0) > 0:
        supabase.table(TABLE).delete().neq("question_id", None).execute()

    # one shot load all records in the csv 
    records = pd.read_csv(CSV_PATH).to_dict(orient="records")
    supabase.table(TABLE).insert(records).execute()

    print(f"Loaded {len(records)} rows into `{TABLE}`.")
    return len(records)

In [ ]:
reload_table_from_csv("quiz_df", "quiz_df.csv")

Loaded 11260 rows into `quiz_df`.


11260

In [47]:
reload_table_from_csv("test_df", "test_df.csv")

Loaded 10000 rows into `test_df`.


10000

In [ ]:
# SELECT * FROM quiz_df LIMIT 10;
quiz_df_supabase = pd.DataFrame(supabase.table('quiz_df').select('*').limit(10).execute().data)
quiz_df_supabase

,question_id,quiz_id,question_text,subtopic,student_id,module_id,module_name,attempt_number,quiz_score_percent,quiz_timestamp,quiz_duration_secs,is_correct
0,QN1,QZ01,How does exploratory data analysis enhance dec...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,1,40,2024-01-13T10:00:48+00:00,521,True
1,QN1,QZ01,How does exploratory data analysis enhance dec...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,2,30,2024-02-22T19:51:22+00:00,1549,False
2,QN1,QZ01,How does exploratory data analysis enhance dec...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,3,60,2024-01-16T03:24:40+00:00,1736,True
3,QN10,QZ01,How can anomaly detection in datasets improve ...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,1,40,2024-01-13T10:00:48+00:00,521,True
4,QN10,QZ01,How can anomaly detection in datasets improve ...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,2,30,2024-02-22T19:51:22+00:00,1549,True
5,QN10,QZ01,How can anomaly detection in datasets improve ...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,3,60,2024-01-16T03:24:40+00:00,1736,True
6,QN2,QZ01,How can outlier detection in data analytics en...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,1,40,2024-01-13T10:00:48+00:00,521,False
7,QN2,QZ01,How can outlier detection in data analytics en...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,2,30,2024-02-22T19:51:22+00:00,1549,False
8,QN2,QZ01,How can outlier detection in data analytics en...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,3,60,2024-01-16T03:24:40+00:00,1736,True
9,QN3,QZ01,How can data anomalies discovered during explo...,Intro to Data Analytics,S01,DAVA,Data Visualization & Analytics,1,40,2024-01-13T10:00:48+00:00,521,False


In [ ]:
# SELECT * FROM test_df LIMIT 10;
test_df_supabase = pd.DataFrame(supabase.table('test_df').select('*').limit(10).execute().data)
test_df_supabaseSho

,question_id,test_id,subtopic,question_text,student_id,module_id,module_name,test_score_percent,test_timestamp,is_correct
0,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S01,DAVA,Data Visualization & Analytics,76,2024-10-17T20:31:13+00:00,True
1,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S02,DAVA,Data Visualization & Analytics,64,2024-01-11T19:10:02+00:00,True
2,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S03,DAVA,Data Visualization & Analytics,76,2024-02-26T19:09:42+00:00,True
3,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S04,DAVA,Data Visualization & Analytics,92,2024-11-28T11:33:06+00:00,True
4,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S05,DAVA,Data Visualization & Analytics,64,2024-01-01T20:51:22+00:00,True
5,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S06,DAVA,Data Visualization & Analytics,56,2024-02-09T08:39:54+00:00,True
6,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S07,DAVA,Data Visualization & Analytics,84,2024-06-28T09:20:40+00:00,True
7,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S08,DAVA,Data Visualization & Analytics,56,2024-09-02T21:02:46+00:00,True
8,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S09,DAVA,Data Visualization & Analytics,56,2024-07-11T05:51:30+00:00,True
9,QN1,TS01,Intro to Data Analytics,How can anomaly detection in data analytics tr...,S10,DAVA,Data Visualization & Analytics,72,2024-01-28T03:41:12+00:00,True


In [49]:
# SELECT COUNT(*) FROM quiz_df;
count = supabase.table('quiz_df').select('*', count='exact', head=True).execute().count
print(count)

assert count == len(quiz_df), "Length of quiz_df in Supabase does not match test_df"

# SELECT COUNT(*) FROM test_df;
count = supabase.table('test_df').select('*', count='exact', head=True).execute().count
print(count)

assert count == len(test_df), "Length of test_df in Supabase does not match test_df"

11260
10000


## Generating embeddings for questions

**What are embeddings?**  
Embeddings are numeric “fingerprints” of text, where each piece of text (e.g. a quiz question) is converted into a vector of numbers. These vectors capture the meaning of the text—similar ideas end up with similar vectors.

**Why use them here?**  
- **Fast, semantic search:** When a student’s query is turned into an embedding, we can quickly find the stored question-embeddings that are closest in meaning, even if they use different words.  
- **Precision:** Embeddings go beyond keyword matching, surfacing questions that are truly relevant to the student’s intent.  
- **Scale in Supabase:** With pgvector, we store and compare thousands of question-embeddings directly in our Postgres database.  
- **RAG pipelines:** Retrieved question texts become context for our LLM explainers, so the model’s answers are accurate and grounded in actual quiz content.

Many modern text embedding models can encode text from 384 to 3072 dimensions. A higher dimensionality allows for more nuance in representing text, but also requires more storage and processing power. A 1536-dimension model would represent each chunk of text as a 1536-dimensional vector (think of it as a vector with 1536 numbers in it. )

An example of different concepts in a (reduced) 3-dimensional space:

<img src="https://pypi-camo.freetls.fastly.net/9735485ae672127dd8fbeb765d7bb1a3b34f816f/68747470733a2f2f7261772e67697468756275736572636f6e74656e742e636f6d2f726f626572742d6d636465726d6f74742f656d62656464696e67735f706c6f742f6d61696e2f696d616765732f6578616d706c6533642e706e67" width="750">

Notice how words with similar meaning end up close to each other, while words with different meanings are further apart. 
<img src="https://cdn.prod.website-files.com/6752ed91ddf74c98d49aa0e4/679b99513ee0c22a8b781001_679a85663fadbc11c9750153_f7044611-93e0-49e5-8b72-765614d9a56d_vectorspace_sentences.webp" width="750">

By performing a semantic search, we can find the closest question-embeddings to a particular question faced by a student, even if it uses different words. The searched can be performed on a large corpus (collection) of slide decks, web articles, or YouTube video transcripts. 

<img src="https://cdn.prod.website-files.com/6752ed91ddf74c98d49aa0e4/679b99503ee0c22a8b780ffb_679a85663fadbc11c975009e_c3e401c7-b95e-4d4b-99e3-58e5cefd0814_semanticqa.webp" width="750">


### Creating unique views
As the quiz_df and test_df have a lot of redundant rows and columns, we need to create views (think of is as virtual SQL tables) with unique question_id. Each embedding will contain information about the question, the subtopic, the module and whethe the question was answered correctly or not. This should provide sufficient information for the LLM to find relevant materials and generate good explanations. 

We can use the following code to generate the unique views for both quiz_df and test_df: 
```
DROP VIEW IF EXISTS distinct_quiz_df;
DROP VIEW IF EXISTS distinct_test_df;

CREATE OR REPLACE VIEW distinct_quiz_df AS
SELECT DISTINCT question_id, question_text, subtopic, module_name
FROM quiz_df;

CREATE OR REPLACE VIEW distinct_test_df AS 
SELECT DISTINCT question_id, question_text, subtopic, module_name
FROM test_df;

SELECT
  (SELECT COUNT(*) FROM distinct_quiz_df) AS count_distinct_quiz_df,
  (SELECT COUNT(*) FROM distinct_test_df) AS count_distinct_test_df;
  ```


Also, Supabase requires a pgvector column (remember the pgvector extension we installed in Postgres?) to store each embedding. We store this in separate quiz_embeddings and test_embeddings tables. This allows for the embeddings to be stored without duplication, saving database storage space. 

We also use IVFFlat index to partition the vectors into groups. During the search, we can use the index to find the closest vectors to the query vector. This is much faster than a brute-force search. 

We can use the following code to create the quiz_embeddings and test_embeddings tables, both of which contain pgvector columns: 
```
-- Create Quiz embeddings table
CREATE TABLE IF NOT EXISTS quiz_embeddings (
  question_id   text        PRIMARY KEY,
  embedding     vector(1536)
);
CREATE INDEX IF NOT EXISTS idx_quiz_emb_ivf
  ON quiz_embeddings
  USING ivfflat (embedding vector_cosine_ops)
  WITH (lists = 100);


-- Create Test embeddings table
CREATE TABLE IF NOT EXISTS test_embeddings (
  question_id   text        PRIMARY KEY,
  embedding     vector(1536)
);
CREATE INDEX IF NOT EXISTS idx_test_emb_ivf
  ON test_embeddings
  USING ivfflat (embedding vector_cosine_ops)
  WITH (lists = 100);


-- Insert unique quiz questions into embeddings tables
INSERT INTO quiz_embeddings (question_id)
SELECT DISTINCT question_id
FROM quiz_df
ON CONFLICT (question_id) DO NOTHING;


-- Insert unique test questions into embeddings tables
INSERT INTO test_embeddings (question_id)
SELECT DISTINCT question_id
FROM test_df
ON CONFLICT (question_id) DO NOTHING;

  ```

In [ ]:
import os
from openai import OpenAI   
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import clear_output

client   = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

views = {
    "distinct_quiz_df":  "quiz_embeddings",
    "distinct_test_df":  "test_embeddings",
}

# Fetch all question_ids + texts
tasks = []
for view, emb_table in views.items():
    resp = supabase.table(view) \
        .select("question_id,question_text,subtopic,module_name") \
        .execute()
    for r in resp.data:
        txt = (
            f"{r['question_text']}\n"
            f"Subtopic: {r['subtopic']}\n"
            f"Module: {r['module_name']}\n"
        )
        tasks.append((emb_table, r["question_id"], txt))

def embed_and_store(table, qid, text):
    vec = client.embeddings.create(model="text-embedding-3-small", input=text) \
               .data[0].embedding
    supabase.table(table) \
        .update({"embedding": vec}) \
        .eq("question_id", qid) \
        .execute()

# parallelize
success = fail = 0
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(embed_and_store, tbl, qid, txt): (tbl, qid)
               for tbl, qid, txt in tasks}
    for fut in as_completed(futures):
        tbl, qid = futures[fut]
        try:
            fut.result(); success += 1
        except:
            fail += 1

In [87]:
print("Count of quiz_embeddings: ", supabase.table('quiz_embeddings').select('embedding', count='exact', head=True).execute().count)
print("Count of test_embeddings: ", supabase.table('test_embeddings').select('embedding', count='exact', head=True).execute().count)
# all 400 embeddings were updated

Count of quiz_embeddings:  200
Count of test_embeddings:  200


In [111]:
import os
import fitz  # PyMuPDF for PDFs
import docx

def extract_docx_text(path):
    doc = docx.Document(path)
    lines = []
    for p in doc.paragraphs:
        if p.text:
            lines.append(p.text)
    return "\n".join(lines)

def extract_pdf_text(path):
    texts = []
    with fitz.open(path) as pdf:
        for page in pdf:
            texts.append(page.get_text("text"))
    return "\n".join(texts)

def chunk_text(text, chunk_size=225, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

def load_and_chunk(base_dir, module_id):
    results = []
    for root, _, files in os.walk(base_dir):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            # skip hidden or ms office temporary cache files
            if ext not in (".docx", ".pdf") or fname.startswith(".") or fname.startswith("~$"):
                continue
            path = os.path.join(root, fname)
            if ext == ".docx":
                full_text = extract_docx_text(path)
            else:
                full_text = extract_pdf_text(path)
            for idx, chunk in enumerate(chunk_text(full_text), 1):
                results.append({
                    "module_id": module_id,
                    "document_name": fname,
                    "chunk_index": idx,
                    "chunk": chunk
                })
    return results

def prepare_for_embedding(chunks, category="Lesson material"):
    payloads = []
    for item in chunks:
        module_id     = item["module_id"]
        document_name = item["document_name"].split(".")[0]
        chunk_index   = item["chunk_index"]
        content       = item["chunk"]
        chunk_id      = f"{document_name}__{chunk_index}"
        payloads.append({
            "chunk_id":      chunk_id,
            "module_id":     module_id,
            "document_name": document_name,
            "category":      category,
            "chunk_index":   chunk_index,
            "content":       content
        })
    return payloads

# Usage
categories = ["DAVA", "LOMA", "DSES", "DWBI"]
all_requests = []

for category in categories:
    folder_path = f"./{category} notes"
    chunks = load_and_chunk(folder_path, category)
    requests = prepare_for_embedding(chunks, category="Lesson material")
    print(f"[{category}] Prepared {len(requests)} records")
    all_requests.extend(requests)

print(f"Total records prepared: {len(all_requests)}")


# `requests` is a list of (module_id, doc_name, chunk_index, input_text) ready for your embedding calls.

[DAVA] Prepared 114 records
[LOMA] Prepared 160 records
[DSES] Prepared 40 records
[DWBI] Prepared 53 records
Total records prepared: 367


In [112]:
all_requests[0]  # sample requests

{'chunk_id': 'P1 Intro to Data Analytics - Student(4)__1',
 'module_id': 'DAVA',
 'document_name': 'P1 Intro to Data Analytics - Student(4)',
 'category': 'Lesson material',
 'chunk_index': 1,
 'content': 'PRACTICAL QUESTIONS FOR DATA VISUALISATION AND ANALYTICS (CIA1C11) Practical 1 – Introduction to Data Analytics / Getting the Required Software This practical aims to introduce students to the data-driven decision approach and orientate students to the KNIME desktop and how it can be used to support the CRISP-DM approach to data analytics. Question 1 (Case Study on Data Literacy) Form a group of 4 to 6 members. Explore the dashboards by going to this website - https://demos.qlik.com/qliksense/CampaignforDataLiteracy Click on Launch Demo. This may take a little while. Once the demo is launched, you will see a selection of dashboards which present the findings of a Data Literacy Survey. Select the dashboard on “Demographics”. Answer the following questions: How many individuals were su

### Creating document chunks

This allows for the documents to be searched against using a method known as retrieval augmented generation.

Retrieval-Augmented Generation (RAG) works by first converting your query into a vector, then using that vector to fetch the most relevant pieces of text (from your documents, slides, transcripts, etc.). Those retrieved passages are combined with your original query and sent to a language model, which generates an answer grounded in the specific, up-to-date content—so you get responses that are both accurate and backed by real data.

 Here are some features of RAG:

- **Fits model limits:** Large documents often exceed token or input-size restrictions of embedding and LLM APIs. Chunks of ~200 words stay safely within those limits.  

- **Improves relevance:** Embedding smaller, self-contained passages yields more precise similarity matches—so your search returns the most pertinent snippet, not an entire 50-page deck.  
- **Enables overlap:** A 50-word overlap between chunks preserves context at the boundaries, preventing important sentences from being split and misunderstood.  
- **Reduces noise:** You avoid diluting embeddings with unrelated content—each chunk centers on a coherent idea or slide.  
- **Speeds retrieval:** Vector indexing on dozens of smaller chunks is far faster and more memory-efficient than indexing entire documents.  

The following SQL query creates the document chunk table, along with the vector embeddings. The indexing function allows for faster vector similarity search.

```
CREATE TABLE doc_chunks (
  chunk_id       TEXT        PRIMARY KEY,       -- e.g. "P1_Intro_to_Data_Analytics_Student4__1"
  module_id      TEXT        NOT NULL,          -- e.g. "DAVA"
  document_name  TEXT        NOT NULL,          -- e.g. "P1 Intro to Data Analytics - Student(4)"
  category       TEXT        NOT NULL,          -- e.g. "Lesson material"
  chunk_index    INT         NOT NULL,          -- the 1-based index of this chunk
  content        TEXT        NOT NULL,          -- the ~200-word chunk itself
  embedding      VECTOR(1536)                     -- your pgvector column (once you’ve generated embeddings)
);

-- if you want fast similarity search on these chunks:
CREATE INDEX ON doc_chunks
  USING ivfflat (embedding vector_cosine_ops)
  WITH (lists = 100);

```


In [115]:
def embed_item(item):
    embed_input = (
        f"Module: {item['module_id']}\n"
        f"Document: {item['document_name']}\n"
        f"Chunk index: {item['chunk_index']}\n\n"
        f"{item['content']}"
    )
    resp = client.embeddings.create(model="text-embedding-3-small", input=embed_input)
    vec = resp.data[0].embedding
    rec = item.copy()
    rec["embedding"] = vec
    return rec

records = []
processed = 0
total = len(all_requests)

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(embed_item, item): item for item in all_requests}
    for future in as_completed(futures):
        processed += 1
        try:
            records.append(future.result())
        except Exception as e:
            print(f"❌ Embedding failed for chunk {futures[future]['chunk_id']}: {e}")
        if processed % 16 == 0 or processed == total:
            print(f"🔄 Completed {processed}/{total} embeddings")

🔄 Completed 16/367 embeddings
🔄 Completed 32/367 embeddings
🔄 Completed 48/367 embeddings
🔄 Completed 64/367 embeddings
🔄 Completed 80/367 embeddings
🔄 Completed 96/367 embeddings
🔄 Completed 112/367 embeddings
🔄 Completed 128/367 embeddings
🔄 Completed 144/367 embeddings
🔄 Completed 160/367 embeddings
🔄 Completed 176/367 embeddings
🔄 Completed 192/367 embeddings
🔄 Completed 208/367 embeddings
🔄 Completed 224/367 embeddings
🔄 Completed 240/367 embeddings
🔄 Completed 256/367 embeddings
🔄 Completed 272/367 embeddings
🔄 Completed 288/367 embeddings
🔄 Completed 304/367 embeddings
🔄 Completed 320/367 embeddings
🔄 Completed 336/367 embeddings
🔄 Completed 352/367 embeddings
🔄 Completed 367/367 embeddings


In [127]:
assert all(len(records['embedding']) == 1536 for records in records)

In [130]:
records[0].keys()

dict_keys(['chunk_id', 'module_id', 'document_name', 'category', 'chunk_index', 'content', 'embedding'])

In [133]:
try:
    up = supabase.table("doc_chunks").upsert(records).execute()
    print(f"Inserted {len(up.data)} records")
except Exception as e:
    print("Error during upsert:", e)

Inserted 367 records
